# grads-dict-accumulate-parents — ex2: propagate one node's contributions via BACK_FUNCS

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `grads-dict-accumulate-parents`. Running the final beacon cell reports progress against the `Backprop: grads dict accumulate parents` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: grads dict accumulate parents` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grads-dict-accumulate-parents`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grads-dict-accumulate-parents"
DD_SUBTOPIC = "Backprop: grads dict accumulate parents"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Propagate one node via BACK_FUNCS + accumulate — quick refresher

One reverse-pass step: take `node` (a MiniTensor with `.recipe`), look up the existing `grads[node]` (the upstream gradient), and for each `(argnum, parent)` in `node.recipe.parents` look up the matching back fn in a `BACK_FUNCS: dict[(func, argnum), fn]` table and accumulate the contribution into `grads[parent]`:

```
out_grad = grads[node]
for argnum, parent in node.recipe.parents.items():
    back_fn      = BACK_FUNCS[(node.recipe.func, argnum)]
    contribution = back_fn(out_grad, node.array, *node.recipe.args)
    grads[parent] = grads.get(parent, 0) + contribution     # get-default-0 + add
```

Two pieces compose: the `accumulate_into_grads` rule from ex1 (`get(parent, 0) + g`) AND a dispatcher that picks the right per-arg back fn.

**Exemplar.** For `y = a + b`, `BACK_FUNCS[(add, 0)] = identity`, `BACK_FUNCS[(add, 1)] = identity`. Reverse pass on `y` with `grads[y] = g_y` writes `grads[a] = g_y` and `grads[b] = g_y`. If `a is b` (i.e. `y = a + a`), both contributions land on `a` and the accumulate-rule sums them to `2 * g_y`.

### Exercise 2 — propagate one node's contributions via BACK_FUNCS

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the per-node reverse-pass step: read `grads[node]`, iterate `node.recipe.parents.items()`, dispatch the matching back fn from a `BACK_FUNCS` table, and accumulate each contribution into `grads[parent]` via `get-default-0 + add`.
> Keywords: BACK_FUNCS, dispatcher, grads-dict, reverse-pass, node-propagate
> ```

**KCs targeted:** `grads-dict-accumulate-parents`, `back-funcs-dispatch`

Implement `propagate_node(node, grads, BACK_FUNCS)`. ONE step of the reverse pass. Mutates `grads` in place.

Inputs:
- `node`: a `MiniTensor` with `.recipe` set (a `Recipe` object). The recipe's `.func` is the forward op (`add`, `mul`, etc), `.args` are the forward args, and `.parents` is a `dict[int, MiniTensor]` mapping argnum to the parent MiniTensor.
- `grads`: `dict[MiniTensor, torch.Tensor]`. Must already contain `node` (the upstream gradient for this node, produced by the parent of `node` in the reverse walk).
- `BACK_FUNCS`: `dict[(func, argnum), Callable]`. Each entry is `back_fn(out_grad, node.array, *node.recipe.args) -> contribution_to_that_arg`.

**Algorithm.**
```
out_grad = grads[node]
for argnum, parent in node.recipe.parents.items():
    back_fn      = BACK_FUNCS[(node.recipe.func, argnum)]
    contribution = back_fn(out_grad, node.array, *node.recipe.args)
    grads[parent] = grads.get(parent, 0) + contribution
```

**Two-step composition.** First step looks up the right per-arg back fn from the table (the dispatcher). Second step is the **same** accumulate rule as ex1: `grads.get(parent, 0) + contribution`. Don't overwrite, don't `+=`, don't `KeyError` on first-touch.

**The `y = a + a` stress case.** When the same parent appears at two argnums, the loop visits each separately and BOTH contributions must land on the parent via the accumulator. Argnum 0 and 1 are different keys in `BACK_FUNCS`, but here they happen to be the same fn (identity for add).

Returns `None`.

In [ ]:
def propagate_node(node, grads: dict, BACK_FUNCS: dict) -> None:
    """One reverse-pass step: look up per-arg back fns, accumulate into parent grads."""
    raise NotImplementedError()


def _test_ex2():
    # Build a tiny BACK_FUNCS table over Python-level ops (add, mul, sub).
    # Each back fn takes (out_grad, out_value, *forward_args) and returns d(out)/d(arg).
    def _add_back0(grad_out, value, a, b): return grad_out                  # d(a+b)/da = 1
    def _add_back1(grad_out, value, a, b): return grad_out                  # d(a+b)/db = 1
    def _mul_back0(grad_out, value, a, b): return grad_out * b              # d(a*b)/da = b
    def _mul_back1(grad_out, value, a, b): return grad_out * a              # d(a*b)/db = a
    def _sub_back0(grad_out, value, a, b): return grad_out                  # d(a-b)/da = 1
    def _sub_back1(grad_out, value, a, b): return -grad_out                 # d(a-b)/db = -1

    def fake_add(a, b): return a + b
    def fake_mul(a, b): return a * b
    def fake_sub(a, b): return a - b

    BACK_FUNCS = {
        (fake_add, 0): _add_back0, (fake_add, 1): _add_back1,
        (fake_mul, 0): _mul_back0, (fake_mul, 1): _mul_back1,
        (fake_sub, 0): _sub_back0, (fake_sub, 1): _sub_back1,
    }

    # --- Case 1: y = a + b. propagate_node on y should write grads[a] = grads[b] = grad_out. ---
    a = MiniTensor(t.tensor([1.0, 2.0, 3.0]), requires_grad=True)
    b = MiniTensor(t.tensor([4.0, 5.0, 6.0]), requires_grad=True)
    y_arr = a.array + b.array
    y = MiniTensor(y_arr, requires_grad=True,
                   recipe=Recipe(func=fake_add, args=(a.array, b.array), parents={0: a, 1: b}))
    grads = {y: t.tensor([1.0, 1.0, 1.0])}
    ret = propagate_node(y, grads, BACK_FUNCS)
    assert ret is None, 'must mutate grads in place and return None'
    assert a in grads and b in grads
    assert t.allclose(grads[a], t.tensor([1.0, 1.0, 1.0])), f'grads[a] = {grads[a]}'
    assert t.allclose(grads[b], t.tensor([1.0, 1.0, 1.0])), f'grads[b] = {grads[b]}'

    # --- Case 2: y = a * b. grads[a] = grad_out * b, grads[b] = grad_out * a. ---
    a2 = MiniTensor(t.tensor([2.0, 3.0]), requires_grad=True)
    b2 = MiniTensor(t.tensor([10.0, 20.0]), requires_grad=True)
    y2_arr = a2.array * b2.array
    y2 = MiniTensor(y2_arr, requires_grad=True,
                    recipe=Recipe(func=fake_mul, args=(a2.array, b2.array), parents={0: a2, 1: b2}))
    grads = {y2: t.tensor([1.0, 1.0])}
    propagate_node(y2, grads, BACK_FUNCS)
    assert t.allclose(grads[a2], t.tensor([10.0, 20.0])), f'd/da: {grads[a2]} (expected b)'
    assert t.allclose(grads[b2], t.tensor([2.0, 3.0])),  f'd/db: {grads[b2]} (expected a)'

    # --- Case 3: y = a - b. grads[a] = grad_out, grads[b] = -grad_out. ---
    a3 = MiniTensor(t.tensor([1.0]), requires_grad=True)
    b3 = MiniTensor(t.tensor([1.0]), requires_grad=True)
    y3 = MiniTensor(t.tensor([0.0]), requires_grad=True,
                    recipe=Recipe(func=fake_sub, args=(a3.array, b3.array), parents={0: a3, 1: b3}))
    grads = {y3: t.tensor([5.0])}
    propagate_node(y3, grads, BACK_FUNCS)
    assert t.allclose(grads[a3], t.tensor([5.0]))
    assert t.allclose(grads[b3], t.tensor([-5.0])), 'sub flips sign on arg1'

    # --- THE CRITICAL TEST: y = a + a. Same parent at argnum 0 and 1; contributions sum. ---
    a4 = MiniTensor(t.tensor([7.0]), requires_grad=True)
    y4_arr = a4.array + a4.array
    y4 = MiniTensor(y4_arr, requires_grad=True,
                    recipe=Recipe(func=fake_add, args=(a4.array, a4.array), parents={0: a4, 1: a4}))
    grads = {y4: t.tensor([1.0])}
    propagate_node(y4, grads, BACK_FUNCS)
    assert t.allclose(grads[a4], t.tensor([2.0])), (
        f'y = a + a reverse-pass case: grads[a] must be 2 * grad_out = 2.0, got {grads[a4]} — '
        'did you overwrite instead of accumulating?'
    )

    # --- Pre-existing parent grad: must be ADDED, not overwritten. ---
    a5 = MiniTensor(t.tensor([1.0]), requires_grad=True)
    b5 = MiniTensor(t.tensor([2.0]), requires_grad=True)
    y5 = MiniTensor(t.tensor([3.0]), requires_grad=True,
                    recipe=Recipe(func=fake_add, args=(a5.array, b5.array), parents={0: a5, 1: b5}))
    grads = {y5: t.tensor([1.0]), a5: t.tensor([100.0])}    # a5 already has a prior contribution
    propagate_node(y5, grads, BACK_FUNCS)
    assert t.allclose(grads[a5], t.tensor([101.0])), (
        f'pre-existing grads[a5] must be added to, got {grads[a5]} (expected 101.0)'
    )
    assert t.allclose(grads[b5], t.tensor([1.0]))

    # --- Out-grad not mutated by the propagation. ---
    a6 = MiniTensor(t.tensor([0.0]), requires_grad=True)
    b6 = MiniTensor(t.tensor([0.0]), requires_grad=True)
    y6 = MiniTensor(t.tensor([0.0]), requires_grad=True,
                    recipe=Recipe(func=fake_add, args=(a6.array, b6.array), parents={0: a6, 1: b6}))
    out_grad_original = t.tensor([3.0])
    grads = {y6: out_grad_original}
    propagate_node(y6, grads, BACK_FUNCS)
    assert t.allclose(out_grad_original, t.tensor([3.0])), (
        f'grads[y6] (out_grad) must not be mutated, got {out_grad_original}'
    )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def propagate_node(node, grads: dict, BACK_FUNCS: dict) -> None:
    out_grad = grads[node]
    for argnum, parent in node.recipe.parents.items():
        back_fn = BACK_FUNCS[(node.recipe.func, argnum)]
        contribution = back_fn(out_grad, node.array, *node.recipe.args)
        grads[parent] = grads.get(parent, 0) + contribution
```

**Why the table key is `(func, argnum)`.** Different forward ops have different gradient rules per argument. `add` has the same back fn for arg 0 and arg 1 (both identity). `mul` has DIFFERENT back fns: `arg 0 -> grad_out * b`, `arg 1 -> grad_out * a`. `sub` has `arg 0 -> grad_out` and `arg 1 -> -grad_out`. The `(func, argnum)` key uniquely identifies which back fn to dispatch.

**Why we accumulate, not overwrite.** Two reasons:
- A parent at TWO argnums of the SAME node (like `y = a + a`): the loop visits each separately and both contributions land on `parent` — must sum.
- A parent visited from MULTIPLE downstream nodes (the general DAG case): each prior `propagate_node` call may have already deposited a contribution. The new one adds.

**Why `back_fn(out_grad, node.array, *node.recipe.args)`.** The back fn signature is `(grad_out, value, *forward_args)`. Some back fns need the forward output value (e.g. softmax-back needs the softmax output); some need the forward inputs (e.g. mul-back needs the other operand). Passing all three covers every case the ARENA manual-autograd layer ships.

**Why we look up via `node.recipe.func`, not `node.func`.** MiniTensors don't store the forward function directly — the recipe does. `node.recipe` is the link between the forward tape and the backward dispatch table.

**Compose with topological sort.** The full reverse pass is: topo-sort the DAG, walk it in reverse, call `propagate_node` on each non-leaf node, then copy `grads[leaf]` into `leaf.grad` for each leaf. This drill is the per-node step — the load-bearing inner block.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()